# Feature Engineering & Preprocessing

customerID is only an identifier, so it won't be used for modeling.

The remaining categorical features need to be converted into a format that can be used by scikit-learn.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

df = pd.read_csv('../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv')

In [4]:
import pandas as pd

df = pd.read_csv("../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv")

## Identifier Removal

customerID uniquely identifies each customer and does not provide information about churn behavior.

The column is removed before feature preparation.

In [5]:
df = df.drop('customerID', axis=1)

df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            7043 non-null   object 
 1   SeniorCitizen     7043 non-null   int64  
 2   Partner           7043 non-null   object 
 3   Dependents        7043 non-null   object 
 4   tenure            7043 non-null   int64  
 5   PhoneService      7043 non-null   object 
 6   MultipleLines     7043 non-null   object 
 7   InternetService   7043 non-null   object 
 8   OnlineSecurity    7043 non-null   object 
 9   OnlineBackup      7043 non-null   object 
 10  DeviceProtection  7043 non-null   object 
 11  TechSupport       7043 non-null   object 
 12  StreamingTV       7043 non-null   object 
 13  StreamingMovies   7043 non-null   object 
 14  Contract          7043 non-null   object 
 15  PaperlessBilling  7043 non-null   object 
 16  PaymentMethod     7043 non-null   object 


In [7]:
df['TotalCharges'] = pd.to_numeric(
    df['TotalCharges'],
    errors='coerce'
)

In [8]:
df.isnull().sum()

gender               0
SeniorCitizen        0
Partner              0
Dependents           0
tenure               0
PhoneService         0
MultipleLines        0
InternetService      0
OnlineSecurity       0
OnlineBackup         0
DeviceProtection     0
TechSupport          0
StreamingTV          0
StreamingMovies      0
Contract             0
PaperlessBilling     0
PaymentMethod        0
MonthlyCharges       0
TotalCharges        11
Churn                0
dtype: int64

In [9]:
df = df.dropna()

df.shape

(7032, 20)

## Target Encoding

Churn is converted into a binary variable.

0 = Customer Retained

1 = Customer Churned

In [10]:
df['Churn'] = df['Churn'].map({
    'No': 0,
    'Yes': 1
})

df['Churn'].value_counts()

Churn
0    5163
1    1869
Name: count, dtype: int64

In [11]:
categorical_cols = df.select_dtypes(include='object').columns

categorical_cols

Index(['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
       'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
       'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
       'PaperlessBilling', 'PaymentMethod'],
      dtype='object')

## Categorical Encoding

Categorical variables are transformed into numerical features using one-hot encoding.

The first category is dropped to avoid redundant variables.

In [12]:
df_encoded = pd.get_dummies(
    df,
    columns=categorical_cols,
    drop_first=True
)

In [13]:
print("Original Shape :", df.shape)
print("Encoded Shape  :", df_encoded.shape)

Original Shape : (7032, 20)
Encoded Shape  : (7032, 31)


In [14]:
X = df_encoded.drop('Churn', axis=1)

y = df_encoded['Churn']

In [15]:
print(X.shape)
print(y.shape)

(7032, 30)
(7032,)


## Train-Test Split

An 80/20 split is used for model training and evaluation.

Stratified sampling preserves the original churn distribution.

In [16]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [17]:
print(X_train.shape)
print(X_test.shape)

(5625, 30)
(1407, 30)


In [18]:
X_train.dtypes.value_counts()

bool       26
int64       2
float64     2
Name: count, dtype: int64

In [19]:
X_train = X_train.astype(int)
X_test = X_test.astype(int)

In [20]:
df_encoded.to_csv(
    "../data/processed/customer_churn_encoded.csv",
    index=False
)

In [21]:
import os

os.listdir("../data/processed")

['customer_churn_encoded.csv']

## Save Processed Dataset

The transformed dataset is stored separately to keep preprocessing steps reproducible and avoid repeating data preparation in later notebooks.

In [22]:
print(df_encoded.shape)

df_encoded.head()

(7032, 31)


,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,Churn,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,...,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,1,29.85,29.85,0,False,True,False,False,True,...,False,False,False,False,False,False,True,False,True,False
1,0,34,56.95,1889.50,0,True,False,False,True,False,...,False,False,False,False,True,False,False,False,False,True
2,0,2,53.85,108.15,1,True,False,False,True,False,...,False,False,False,False,False,False,True,False,False,True
3,0,45,42.30,1840.75,0,True,False,False,False,True,...,False,False,False,False,True,False,False,False,False,False
4,0,2,70.70,151.65,1,False,False,False,True,False,...,False,False,False,False,False,False,True,False,True,False


In [23]:
df_encoded.isnull().sum().sum()

0

## Train-Test Split

The dataset is split into training and testing samples using an 80/20 ratio.

Stratification is applied to maintain the original churn distribution.

In [24]:
from sklearn.model_selection import train_test_split

X = df_encoded.drop("Churn", axis=1)
y = df_encoded["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [25]:
X_train.shape
X_test.shape
X_train.dtypes.value_counts()

bool       26
int64       2
float64     2
Name: count, dtype: int64

In [26]:
print("Training Set:", X_train.shape)
print("Testing Set :", X_test.shape)

Training Set: (5625, 30)
Testing Set : (1407, 30)


## Preprocessing Summary

Completed:

- Removed customer identifier
- Converted TotalCharges to numeric format
- Encoded churn target
- Applied one-hot encoding
- Created training and testing datasets

The data is ready for model training.

In [27]:
import pandas as pd

df = pd.read_csv(
    "../data/processed/customer_churn_encoded.csv"
)

df.shape

(7032, 31)